# Training of the Z-band 3D U-Net

`sarc.motion.detect_z_bands_fast_movie` predicts Z-bands with a 3D U-Net that looks at several frames at once. Its
Z-band masks flicker much less than the per-frame 2D output, which halves the frame-to-frame noise of the tracked
sarcomere lengths, but only on data like what it was trained on. This tutorial retrains or fine-tunes it on your
own movies, with the four scripts in `scripts/retrain_unet3d/` that trained the bundled model (code in
`sarcasm/training/`).

**The 3D model is meant for high-speed recordings only**: about 30 frames per second or faster, so that a
contraction spans at least ~6 frames, at high magnification, with a sarcomere about 10–32 px long. For slower or
lower-magnification movies, use `detect_sarcomeres` (2D). Training needs a GPU and takes a few hours.

## Where the labels come from

Nobody labels movies by hand. Two sources are combined:

| | what | labels |
|---|---|---|
| **synthetic clips** (step 1) | images of the 2D training set, animated: contraction along the myofibrils, drift, frame-to-frame jitter, camera noise | exact: the label is warped with the image |
| **teacher clips** (step 2) | short windows cut from real high-speed movies | the 2D U-Net, run on flow-denoised frames and averaged over neighbouring frames |

The synthetic clips teach the model motion and noise it never saw in real movies; the teacher clips teach it what
real data from your microscope look like. The teacher labels are only as good as the 2D model: look at a few
teacher clips (e.g. in napari) before training. If the 2D model is wrong on your data, retrain it first
([2D U-Net tutorial](tutorial_training_unet.ipynb)).

In [ ]:
scripts = '../../scripts/retrain_unet3d'      # adjust paths
training_data_2d = 'path/to/training_data_2d'  # the 2D training set: image/, zbands/, ... (see 2D U-Net tutorial)
clips = 'path/to/clips'

## 1. Synthetic clips

From the 2D training set (CPU, minutes). The validation clips come from the images of the validation split.

In [ ]:
!python {scripts}/1_make_synthetic_clips.py {training_data_2d} {clips}/synthetic --n 4000
!python {scripts}/1_make_synthetic_clips.py {training_data_2d} {clips}/synthetic_val --n 100 --split val

## 2. Teacher clips

From real movies (~1 min per movie on a GPU). List them in a CSV with the columns `path,pixelsize,name` (`name`:
short and unique, it prefixes the clip files):

```
path,pixelsize,name
/data/our_lab/Series004.tif,0.0659,ours_s004
/data/collaborator/movie_051.tif,0.108,collab_051
```

Hold out **whole movies** for validation, in a second CSV: neighbouring frames are nearly identical, so a split
by window measures memory, not generalisation.

In [ ]:
!python {scripts}/2_make_teacher_clips.py movies.csv {clips}/real
!python {scripts}/2_make_teacher_clips.py movies_val.csv {clips}/real_val

## 3. Training

`NAME=FOLDER:WEIGHT` sets how often training patches come from each folder; the weights, not the clip counts, set
the mix; a folder marked `:real` holds teacher clips. `--time_weight` penalises frame-to-frame flicker (default
0). Without it the sarcomere trajectories come out about twice as noisy; the bundled model was fine-tuned with 3. The penalty also smooths real fast events (sarcomere
popping), so check both on a tracked movie before choosing. To fine-tune the bundled or your own model instead:
`--init model.pt --epochs 15 --lr 1e-4`.

In [ ]:
!python {scripts}/3_train.py --out runs/my_model --time_weight 3 \
    --data synthetic={clips}/synthetic:0.5 --data real={clips}/real:0.5:real \
    --val synthetic={clips}/synthetic_val --val real={clips}/real_val

## 4. Evaluation

Compare your model with the bundled models and with the per-frame 2D prediction, on the held-out clips. `fast_*`
scores the fastest-moving frames.

In [ ]:
!python {scripts}/4_evaluate.py {clips}/synthetic_val {clips}/real_val \
    --model mine=runs/my_model/best.pt --model generalist --model legacy --with_2d 0.1

Then use the model:

```python
sarc.motion.detect_z_bands_fast_movie(model_path='runs/my_model/best.pt')
```

and check it on a tracked movie: the frame-to-frame noise of the sarcomere lengths and the number of popping
events (see the [tracking tutorial](tutorial_sarcomere_tracking.ipynb)).